In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import importlib.util, json, sys, time, zipfile
from pathlib import Path
from collections import OrderedDict
import numpy as np
import torch
import torch.nn.functional as F

assert torch.cuda.is_available(), 'Select a GPU runtime before running this cell.'
device = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0), flush=True)

DRIVE_ROOT = Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/03_training_ready')
BUNDLE_ZIP = DRIVE_ROOT / 'phase7_smoke_bundle.zip'
BUNDLE_DIR = Path('/content/phase7_smoke_bundle')
TRAINER_PATH = BUNDLE_DIR / 'experiments/phase5_train_temporal.py'
if not TRAINER_PATH.is_file():
    if not BUNDLE_ZIP.is_file():
        raise FileNotFoundError(f'Missing source bundle: {BUNDLE_ZIP}')
    BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(BUNDLE_ZIP) as zf:
        zf.extractall(BUNDLE_DIR)
sys.path.insert(0, str(BUNDLE_DIR))
spec = importlib.util.spec_from_file_location('phase5_train_temporal_smoke', TRAINER_PATH)
trainer = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = trainer
spec.loader.exec_module(trainer)

CACHE_DIR = DRIVE_ROOT / 'checkpoints/phase7_capture_holdout/precomputed_inputs/1e72186b7149de228ddd9cba'
manifest_path = CACHE_DIR / 'cache_manifest.json'
if not manifest_path.is_file():
    raise FileNotFoundError(f'Cache manifest missing: {manifest_path}')
manifest = json.loads(manifest_path.read_text())
completed = sorted(int(x) for x in manifest.get('completed_shards', []))
if not completed:
    completed = sorted(int(p.stem.split('_')[-1]) for p in CACHE_DIR.glob('train_shard_*.npz'))
if not completed:
    raise RuntimeError('No completed cache shards are present yet.')
shard_id = completed[0]
print(f'Cache manifest complete={manifest.get("complete")}; available shards={len(completed)}/{manifest.get("shard_count", "?")}; testing shard {shard_id}', flush=True)

# Exercise the same shard reader used by the trainer, bypassing only its intentional
# complete-cache guard because this is a bounded test of already-finished shards.
store = object.__new__(trainer.PrecomputedSampleStore)
store.root = CACHE_DIR.resolve()
store.manifest = manifest
store.shard_windows = int(manifest['shard_windows'])
store.variants = int(manifest['variants'])
store.image_size = int(manifest['image_size'])
store._cache = OrderedDict()
t0 = time.perf_counter()
shard = store._load_shard(shard_id)
read_seconds = time.perf_counter() - t0
print('Shard arrays:', {k: tuple(v.shape) for k, v in shard.items() if hasattr(v, 'shape')}, flush=True)
print(f'First Drive shard load + decompress: {read_seconds:.2f}s; shard size on Drive: {(CACHE_DIR / f"train_shard_{shard_id:05d}.npz").stat().st_size/1024**2:.1f} MiB', flush=True)

BATCH = min(16, len(shard['x']))
if BATCH < 2:
    raise RuntimeError(f'Completed shard has only {BATCH} windows; need at least 2.')
def fetch_batch(start):
    rows = [store.get(shard_id * store.shard_windows + ((start + j) % len(shard['x'])), j % store.variants) for j in range(BATCH)]
    return tuple(torch.from_numpy(np.stack([row[k] for row in rows])) for k in range(6))

# Warm a batch, then include CPU batch assembly, host-to-device copy, feature packing,
# forward, loss, backward, and optimizer step in the timed training-step measurement.
model = trainer.TemporalStateCorrector(hidden_dim=256).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scaler = torch.amp.GradScaler('cuda', enabled=True)
scales = torch.tensor([0.05]*3 + [0.34906585]*3 + [0.52359878]*21, device=device).view(1,1,27)

def train_step(batch_start):
    cpu = fetch_batch(batch_start)
    x, states, target_delta, valid, dt, gate_target = [v.to(device, non_blocking=True) for v in cpu]
    features = trainer.batch_features(states, device)
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=True):
        out = model.forward_sequence(x, features, dt, frame_valid=valid)
        pred = torch.cat((out.delta_T, out.delta_R, out.delta_q), dim=-1)
        applied = pred * out.authority[..., None]
        err = (applied - target_delta) / scales
        vw = valid[..., None]
        denom = vw.sum().clamp_min(1.0) * pred.shape[-1]
        pose = (err.square() * vw).sum() / denom
        logsig = out.log_sigma.float().clamp(-5.0, 2.0)
        nll = err.float().square() * torch.exp(-2 * logsig) + 2 * logsig
        unc = (nll * vw).sum() / denom
        with torch.autocast(device_type='cuda', enabled=False):
            gate = F.binary_cross_entropy(out.authority.float().clamp(1e-5, 1-1e-5), gate_target.float())
        pair = valid[:, 1:] * valid[:, :-1]
        motion = (applied[:, 1:] - applied[:, :-1]) / scales
        smooth = (motion.square().mean(dim=-1) * pair).sum() / pair.sum().clamp_min(1)
        loss = pose + 0.025 * unc + 0.04 * gate + 0.002 * smooth
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
    scaler.step(optimizer)
    scaler.update()
    return float(loss.detach().float().cpu())

for i in range(3):
    train_step(i * BATCH)
torch.cuda.synchronize()
step_ms = []
loss_value = None
for i in range(10):
    t0 = time.perf_counter()
    loss_value = train_step(i * BATCH)
    torch.cuda.synchronize()
    step_ms.append((time.perf_counter() - t0) * 1000)
print(f'Cached actual training step (batch={BATCH} windows × {shard["x"].shape[2]} frames; includes batch assembly + H2D + forward/backward): median={float(np.median(step_ms)):.1f} ms; p95={float(np.percentile(step_ms,95)):.1f} ms; loss={loss_value:.5f}', flush=True)
print(f'Peak allocated VRAM: {torch.cuda.max_memory_allocated()/1024**3:.2f} GiB; peak reserved VRAM: {torch.cuda.max_memory_reserved()/1024**3:.2f} GiB', flush=True)
print('This is a bounded throughput smoke test only: no checkpoint written and no cache files or manifest modified.', flush=True)


In [ ]:
# Partial-cache GPU training: exactly one pass over 100 already-complete shards.
# Uses a bounded CPU RAM prefetch queue; does not edit the cache manifest or the main checkpoint.
from google.colab import drive
drive.mount('/content/drive')

import importlib.util, json, os, shutil, sys, time, zipfile
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path
from collections import OrderedDict
import numpy as np
import torch
import torch.nn.functional as F

assert torch.cuda.is_available(), 'This cell requires a GPU runtime.'
device = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0), flush=True)
torch.backends.cuda.matmul.allow_tf32 = True

DRIVE_ROOT = Path('/content/drive/MyDrive/Persistent Visual-to-MANO Hand Tracker/04_DATASET_STAGING/ReInterHand/03_training_ready')
CACHE_DIR = DRIVE_ROOT / 'checkpoints/phase7_capture_holdout/precomputed_inputs/1e72186b7149de228ddd9cba'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints/phase7_capture_holdout'
BUNDLE_ZIP = DRIVE_ROOT / 'phase7_smoke_bundle.zip'
BUNDLE_DIR = Path('/content/phase7_smoke_bundle')
TRAINER_PATH = BUNDLE_DIR / 'experiments/phase5_train_temporal.py'
if not TRAINER_PATH.is_file():
    if not BUNDLE_ZIP.is_file():
        raise FileNotFoundError(f'Missing trainer bundle: {BUNDLE_ZIP}')
    BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(BUNDLE_ZIP) as archive:
        archive.extractall(BUNDLE_DIR)
sys.path.insert(0, str(BUNDLE_DIR))
spec = importlib.util.spec_from_file_location('phase5_partial_cache_train', TRAINER_PATH)
trainer = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = trainer
spec.loader.exec_module(trainer)

manifest_path = CACHE_DIR / 'cache_manifest.json'
manifest = json.loads(manifest_path.read_text())
completed = sorted(int(x) for x in manifest.get('completed_shards', []))
completed = [sid for sid in completed if (CACHE_DIR / f'train_shard_{sid:05d}.npz').is_file()]
if len(completed) < 100:
    raise RuntimeError(f'Need 100 completed shards; found {len(completed)}. No training started.')
selected_shards = completed[:100]
assert len(selected_shards) == 100
print(f'Cache complete={manifest.get("complete")}; completed shards now={len(completed)}/{manifest.get("shard_count")}; training this pass on exactly 100 shards ({selected_shards[0]}..{selected_shards[-1]}), 6,400 windows.', flush=True)
print('This partial pass will stop after one visit to those 100 shards. Validation is not available in these train shards.', flush=True)

# Load the actual Phase 5 architecture and its shard format, but intentionally do
# not construct PrecomputedSampleStore because it rejects an incomplete manifest.
def load_shard(shard_id):
    path = CACHE_DIR / f'train_shard_{int(shard_id):05d}.npz'
    with np.load(path, allow_pickle=False) as saved:
        signature = str(saved['dataset_signature'].item())
        if signature != manifest['dataset_signature']:
            raise RuntimeError(f'Shard signature mismatch: {path.name}')
        data = {key: saved[key].copy() for key in ('x','states','deltas','valid','dt','gate')}
    if data['x'].shape[0] != 64:
        raise RuntimeError(f'Unexpected partial shard length: {path.name}: {data["x"].shape}')
    return data

# Train in randomized shard order, with 20 complete decoded shards buffered in host RAM.
# Each shard is ~120 MiB decoded (all variants); queue target is ~2.4 GiB plus worker overhead.
rng = np.random.default_rng(20251018 + 4100)
shard_order = np.asarray(selected_shards, dtype=np.int64)
rng.shuffle(shard_order)
PREFETCH_SHARDS = 20
LOADER_WORKERS = 4
BATCH_WINDOWS = 32
pending = {}
next_to_schedule = 0

def fill_prefetch(pool):
    global next_to_schedule
    while next_to_schedule < len(shard_order) and len(pending) < PREFETCH_SHARDS:
        sid = int(shard_order[next_to_schedule])
        pending[sid] = pool.submit(load_shard, sid)
        next_to_schedule += 1

try:
    import psutil
    ram = psutil.virtual_memory()
    print(f'RAM before prefetch: available={ram.available/1024**3:.1f} GiB; queue target={PREFETCH_SHARDS} shards; workers={LOADER_WORKERS}.', flush=True)
except Exception:
    print(f'RAM queue target={PREFETCH_SHARDS} shards; workers={LOADER_WORKERS}.', flush=True)

model = trainer.TemporalStateCorrector(hidden_dim=256).to(device)
source_checkpoint = CHECKPOINT_DIR / 'latest.pt'
source_label = 'fresh initialization'
if source_checkpoint.is_file():
    try:
        saved = torch.load(source_checkpoint, map_location='cpu', weights_only=False)
        state = saved.get('model') or saved.get('model_state_dict') or saved.get('state_dict')
        if state is not None:
            model.load_state_dict(state, strict=True)
            source_label = str(source_checkpoint)
    except Exception as exc:
        print(f'Existing checkpoint not used ({type(exc).__name__}: {exc}); starting fresh.', flush=True)
print('Initialization:', source_label, '(main checkpoint will not be overwritten).', flush=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scaler = torch.amp.GradScaler('cuda', enabled=True)
scales = torch.tensor([0.05]*3 + [0.34906585]*3 + [0.52359878]*21, device=device).view(1,1,27)

run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
PARTIAL_DIR = CHECKPOINT_DIR / 'partial_cache100' / run_id
PARTIAL_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_CKPT_DIR = Path('/content/phase7_partial_ckpts')
LOCAL_CKPT_DIR.mkdir(parents=True, exist_ok=True)

def build_batch(data, local_order, start):
    ids = local_order[start:start+BATCH_WINDOWS]
    variants = rng.integers(0, int(manifest['variants']), size=len(ids))
    x = np.stack([data['x'][int(i), int(v)] for i,v in zip(ids,variants)]).astype(np.float32) / 255.0
    states = np.stack([data['states'][int(i), int(v)] for i,v in zip(ids,variants)]).astype(np.float32, copy=False)
    deltas = np.stack([data['deltas'][int(i), int(v)] for i,v in zip(ids,variants)]).astype(np.float32, copy=False)
    valid = np.stack([data['valid'][int(i), int(v)] for i,v in zip(ids,variants)]).astype(np.float32, copy=False)
    dt = np.stack([data['dt'][int(i), int(v)] for i,v in zip(ids,variants)]).astype(np.float32, copy=False)
    gate = np.stack([data['gate'][int(i), int(v)] for i,v in zip(ids,variants)]).astype(np.float32, copy=False)
    return tuple(torch.from_numpy(a).pin_memory() for a in (x,states,deltas,valid,dt,gate))

def one_train_step(cpu_batch):
    x, states, target_delta, valid, dt, gate_target = [v.to(device, non_blocking=True) for v in cpu_batch]
    features = trainer.batch_features(states, device)
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=True):
        out = model.forward_sequence(x, features, dt, frame_valid=valid)
        pred = torch.cat((out.delta_T, out.delta_R, out.delta_q), dim=-1)
        applied = pred * out.authority[..., None]
        err = (applied - target_delta) / scales
        vw = valid[..., None]
        denom = vw.sum().clamp_min(1.0) * pred.shape[-1]
        pose = (err.square() * vw).sum() / denom
        logsig = out.log_sigma.float().clamp(-5.0, 2.0)
        nll = err.float().square() * torch.exp(-2 * logsig) + 2 * logsig
        unc = (nll * vw).sum() / denom
        with torch.autocast(device_type='cuda', enabled=False):
            gate = F.binary_cross_entropy(out.authority.float().clamp(1e-5,1-1e-5), gate_target.float())
        pair = valid[:,1:] * valid[:,:-1]
        motion = (applied[:,1:] - applied[:,:-1]) / scales
        smooth = (motion.square().mean(dim=-1) * pair).sum() / pair.sum().clamp_min(1)
        loss = pose + 0.025*unc + 0.04*gate + 0.002*smooth
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
    scaler.step(optimizer)
    scaler.update()
    return float(loss.detach().float().cpu())

def save_partial(step, shard_id, batch_offset, final=False):
    name = 'final.pt' if final else f'step_{step:04d}.pt'
    local_path = LOCAL_CKPT_DIR / name
    payload = {
        'model': {k:v.detach().cpu() for k,v in model.state_dict().items()},
        'optimizer': optimizer.state_dict(),
        'global_step': int(step), 'shard_id': int(shard_id), 'batch_offset': int(batch_offset),
        'selected_shards': [int(x) for x in selected_shards],
        'dataset_signature': manifest['dataset_signature'],
        'cache_complete_at_start': bool(manifest.get('complete', False)),
        'initialization': source_label, 'seed': 20251018 + 4100,
        'run_id': run_id, 'partial_stage_only': True,
        'torch_rng': torch.get_rng_state(), 'cuda_rng': torch.cuda.get_rng_state_all(),
        'numpy_rng': rng.bit_generator.state,
    }
    torch.save(payload, local_path)
    drive_path = PARTIAL_DIR / name
    shutil.copy2(local_path, drive_path)
    (PARTIAL_DIR / 'latest.txt').write_text(name + chr(10), encoding='utf-8')
    print(f'Checkpoint saved: {drive_path}', flush=True)

start_time = time.perf_counter()
step = 0
loss_window = []
data_wait_s = 0.0
total_batches = 100 * (64 // BATCH_WINDOWS)
with ThreadPoolExecutor(max_workers=LOADER_WORKERS) as pool:
    fill_prefetch(pool)
    # Let the first few shards become resident before beginning the GPU pass.
    for sid in shard_order[:min(8, len(shard_order))]:
        pending[int(sid)].result()
    print(f'RAM prefetch primed with at least {min(8,len(shard_order))} shards; starting GPU pass. VRAM used={torch.cuda.memory_allocated()/1024**3:.2f} GiB.', flush=True)
    for shard_pos, sid_raw in enumerate(shard_order):
        sid = int(sid_raw)
        t_wait = time.perf_counter()
        future = pending.pop(sid)
        data = future.result()
        data_wait_s += time.perf_counter() - t_wait
        fill_prefetch(pool)
        local_order = rng.permutation(len(data['x']))
        for offset in range(0, len(local_order), BATCH_WINDOWS):
            cpu_batch = build_batch(data, local_order, offset)
            loss_value = one_train_step(cpu_batch)
            step += 1
            loss_window.append(loss_value)
            if step % 10 == 0 or step == 1 or step == total_batches:
                torch.cuda.synchronize()
                elapsed = time.perf_counter() - start_time
                rate = step / max(elapsed, 1e-6)
                eta = (total_batches - step) / max(rate, 1e-6)
                try:
                    ram_used = psutil.virtual_memory().used/1024**3
                    ram_text = f'{ram_used:.1f} GiB'
                except Exception:
                    ram_text = 'n/a'
                print(f'[partial100] step {step}/{total_batches} ({100*step/total_batches:.1f}%) loss={loss_value:.5f}; elapsed={elapsed/60:.1f} min; ETA={eta/60:.1f} min; shard={shard_pos+1}/100; data-wait={data_wait_s:.1f}s; RAM used={ram_text}; VRAM={torch.cuda.memory_allocated()/1024**3:.2f} GiB', flush=True)
            if step % 25 == 0:
                save_partial(step, sid, offset)
        del data
    torch.cuda.synchronize()
    save_partial(step, int(shard_order[-1]), 64, final=True)
    print(f'PARTIAL PASS COMPLETE: trained {step} batches over 100 shards / 6,400 windows in {(time.perf_counter()-start_time)/60:.1f} min.', flush=True)
    print(f'Partial-stage checkpoints: {PARTIAL_DIR}', flush=True)
    print('The main latest.pt/update_model.pt and cache manifest were not changed. This partial pass has no validation score; continue full training only after all cache shards are complete.', flush=True)
